# Caso 06: AulaViva — Cursos online: ¿Qué alumno se queda?
## Día 2 (Mañana y Tarde): Antes de Calcular, Limpiar (y Decidir qué es el Valor) + El Ranking de Superficie en SQL

**Objetivos del Día 2:**
1. **Día 2 (Mañana) — Limpieza Forense por las 4 Clases de Suciedad:**
   - **1. Formato y Grafías:**
     - Convertir fechas en 3 formatos (`ISO`, `DD/MM/YYYY` y mes en inglés `"12 Mar 2024"`) con `pd.to_datetime(..., format='mixed', dayfirst=True)`.
     - Desmontar importes europeos (`"5.397,33 €"`) en el orden exacto: fuera símbolo `€`, fuera espacios, **fuera punto de miles (`.`)** y cambio de coma decimal (`,`) a punto (`.`).
     - Mapear con diccionario explícito las **19 grafías de `canal`** a los **5 canales canónicos** (`cupon`, `anuncios`, `seo`, `referido`, `empresas`), así como `plan` (`mensual`, `anual`), `categoria` (6 categorías), `tipo` de pago (`alta`, `renovacion`) y `completado` (`bool`).
   - **2. Duplicados:** Deduplicar por clave primaria en `pagos` (`pago_id`, webhook de Stripe), `matriculas` (`matricula_id`, doble clic) y `suscripciones` (`suscripcion_id`, reintentos).
   - **3. Datos Imposibles (Validar y marcar, NO borrar la fila):** Identificar y marcar importes negativos/cero o no numéricos en `pagos`, avances `> 100 %` o `minutos_vistos` extremos (`999999`) en `matriculas`, y lecciones negativas en `cursos`, excluyéndolos únicamente de las agregaciones que distorsionarían.
   - **4. Nulos con Significado vs. Nulos Reales:** Conservar `campana_id` vacío (alta sin campaña / SEO / Referido $ightarrow$ exige `LEFT JOIN`), conservar `fecha_fin` vacía (suscripción `activa` que sigue vigente) e imputar `metodo` vacío en `pagos` como `'desconocido'`.
2. **La Decisión de Criterio — ¿Qué es el «Valor» de un Canal?:**
   - Definir formalmente la métrica antes de consultar: *«El valor de un canal es el LTV medio de sus alumnos, apoyado en la tasa de finalización y la tasa de renovación, y cerrado con su cuota de valor sobre el total de AulaViva»*.
3. **Día 2 (Tarde) — El Ranking de Superficie en SQL (DuckDB):**
   - Confirmar con SQL las métricas que mira Growth (Diego Losada): **Altas por canal** (con función de ventana `SUM(COUNT(*)) OVER ()`), **CAC por canal** (mediante **CTEs separadas** para no multiplicar filas), **Ingreso de primeras cuotas** (la primera grieta: gana `anuncios`, no `cupon`) y **Horas totales de vídeo**.
   - Clasificar **Contexto** (matrículas por categoría) y **Ruido** (mix de plan mensual/anual idéntico ~68% / 32% en todos los canales).


In [1]:
import pandas as pd
import numpy as np
import os
import duckdb

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

RAW_DIR = os.path.join('data', 'raw')
CLEAN_DIR = os.path.join('data', 'clean')
os.makedirs(CLEAN_DIR, exist_ok=True)

def limpiar_moneda(serie):
    """
    Desmonta importes europeos ('5.397,33 €') en el orden correcto:
    1) Quitar símbolo € y espacios
    2) Quitar punto de miles (.)
    3) Cambiar coma decimal (,) por punto (.)
    4) Convertir a float (valores como 'N/D', 'error', '-' pasan a NaN)
    """
    s = (
        serie.astype(str)
        .str.replace('€', '', regex=False)
        .str.replace(' ', '', regex=False)
        .str.replace('.', '', regex=False)
        .str.replace(',', '.', regex=False)
        .str.strip()
    )
    return pd.to_numeric(s, errors='coerce')

def limpiar_porcentaje_100(serie):
    """Convierte texto como '87 %' a número en escala 0-100 (87.0)."""
    s = (
        serie.astype(str)
        .str.replace('%', '', regex=False)
        .str.replace(' ', '', regex=False)
        .str.replace(',', '.', regex=False)
        .str.strip()
    )
    return pd.to_numeric(s, errors='coerce')

def limpiar_horas(serie):
    """Convierte texto como '10,0 h' a float (10.0)."""
    s = (
        serie.astype(str)
        .str.replace('h', '', regex=False)
        .str.replace(' ', '', regex=False)
        .str.replace(',', '.', regex=False)
        .str.strip()
    )
    return pd.to_numeric(s, errors='coerce')

def parsear_fecha(serie):
    """Convierte los 3 formatos (ISO, DD/MM/YYYY y mes en inglés '12 Mar 2024') a datetime64."""
    return pd.to_datetime(serie, format='mixed', dayfirst=True, errors='coerce')

MAPA_CANAL = {
    'cupon': 'cupon',
    'cupón': 'cupon',
    'descuento': 'cupon',
    'anuncios': 'anuncios',
    'ads': 'anuncios',
    'publicidad': 'anuncios',
    'seo': 'seo',
    'organico': 'seo',
    'orgánico': 'seo',
    'buscador': 'seo',
    'referido': 'referido',
    'amigo': 'referido',
    'recomendacion': 'referido',
    'recomendación': 'referido',
    'empresas': 'empresas',
    'b2b': 'empresas',
    'corporativo': 'empresas'
}

MAPA_PLAN = {
    'mensual': 'mensual',
    'mes': 'mensual',
    'anual': 'anual',
    'año': 'anual',
    '12m': 'anual'
}


### 1. Limpieza de `canales.csv` y `campanas.csv`
* En `canales.csv`: normalizamos `nombre` y `tipo` a minúsculas limpias.
* En `campanas.csv`: normalizamos `canal` con el diccionario explícito `MAPA_CANAL`, limpiamos espacios en `nombre`, convertimos `coste` a `float` (desmontando primero el punto de miles) y parseamos `fecha_inicio` y `fecha_fin`.


In [2]:
# 1A. canales.csv
df_canales = pd.read_csv(os.path.join(RAW_DIR, 'canales.csv'), encoding='utf-8')
df_canales['nombre'] = df_canales['nombre'].astype(str).str.strip().str.lower()
df_canales['tipo'] = df_canales['tipo'].astype(str).str.strip().str.lower()

# 1B. campanas.csv
df_campanas = pd.read_csv(os.path.join(RAW_DIR, 'campanas.csv'), encoding='utf-8')
df_campanas['canal_raw'] = df_campanas['canal']
df_campanas['canal'] = df_campanas['canal'].astype(str).str.strip().str.lower().map(MAPA_CANAL)
df_campanas['nombre'] = df_campanas['nombre'].astype(str).str.strip()
df_campanas['coste'] = limpiar_moneda(df_campanas['coste'])
df_campanas['fecha_inicio'] = parsear_fecha(df_campanas['fecha_inicio'])
df_campanas['fecha_fin'] = parsear_fecha(df_campanas['fecha_fin'])
df_campanas['tipo'] = df_campanas['tipo'].astype(str).str.strip().str.lower()
df_campanas = df_campanas.drop(columns=['canal_raw'])

print("Verificación de canales en campanas.csv:", df_campanas['canal'].value_counts(dropna=False).to_dict())
print("Coste total por canal en campanas.csv:")
print(df_campanas.groupby('canal')['coste'].agg(['count', 'sum']).round(2))


Verificación de canales en campanas.csv: {'cupon': 18, 'anuncios': 12, 'empresas': 8, 'referido': 7}
Coste total por canal en campanas.csv:
          count       sum
canal                    
anuncios     12  211200.0
cupon        18   91200.0
empresas      8   96000.0
referido      7   48000.0


### 2. Limpieza de `usuarios.csv` (Las 19 grafías de canal a 5 y el Nulo con Significado en `campana_id`)
* Mapeamos las **19 grafías de `canal`** a los **5 canales canónicos** (`cupon`, `anuncios`, `seo`, `referido`, `empresas`) y comprobamos que no quede ningún `NaN` (`nunique == 5`).
* Mapeamos `plan_inicial` a `mensual` / `anual` y `ciudad` a las 5 ciudades (`Valencia`, `Madrid`, `Barcelona`, `Sevilla`, `Bilbao`, o `Desconocida` si era nulo).
* **Nulo con significado (`campana_id`):** Conservamos los `14.255` vacíos en `campana_id` (tipo entero nullable `Int64`) y añadimos la bandera explícita `sin_campana` (`True` cuando `campana_id` es nulo).


In [3]:
MAPA_CIUDAD = {
    'valencia': 'Valencia', 'vlc': 'Valencia',
    'madrid': 'Madrid', 'mad': 'Madrid',
    'barcelona': 'Barcelona', 'bcn': 'Barcelona',
    'sevilla': 'Sevilla', 'svq': 'Sevilla',
    'bilbao': 'Bilbao', 'bio': 'Bilbao'
}

df_usuarios = pd.read_csv(os.path.join(RAW_DIR, 'usuarios.csv'), encoding='utf-8')
df_usuarios = df_usuarios.drop_duplicates(subset=['usuario_id']).copy()

df_usuarios['canal'] = df_usuarios['canal'].astype(str).str.strip().str.lower().map(MAPA_CANAL)
assert df_usuarios['canal'].isna().sum() == 0, "¡Alerta! Quedó alguna grafía de canal sin mapear"
assert df_usuarios['canal'].nunique() == 5, "¡Alerta! No salieron exactamente 5 canales"

df_usuarios['plan_inicial'] = df_usuarios['plan_inicial'].astype(str).str.strip().str.lower().map(MAPA_PLAN)
df_usuarios['ciudad'] = (
    df_usuarios['ciudad']
    .astype(str).str.strip().str.lower()
    .map(MAPA_CIUDAD)
    .fillna('Desconocida')
)
df_usuarios['fecha_registro'] = parsear_fecha(df_usuarios['fecha_registro'])
df_usuarios['campana_id'] = df_usuarios['campana_id'].astype('Int64')
df_usuarios['sin_campana'] = df_usuarios['campana_id'].isna()

print("Recuento limpio de usuarios por canal (19 grafías -> 5 canales):")
print(df_usuarios['canal'].value_counts())
print("\nFechas sin parsear (NaT):", df_usuarios['fecha_registro'].isna().sum())


Recuento limpio de usuarios por canal (19 grafías -> 5 canales):
canal
cupon       15200
anuncios     9600
seo          8000
referido     4800
empresas     2400
Name: count, dtype: int64

Fechas sin parsear (NaT): 0


### 3. Limpieza de `suscripciones.csv` (Deduplicación y `fecha_fin` vacía = Suscripción Activa)
* **Deduplicación:** Eliminamos los `358` duplicados por `suscripcion_id` (`51.601` $\rightarrow$ `51.243` suscripciones únicas).
* **Nulo con significado (`fecha_fin`):** Las suscripciones con `fecha_fin` vacía son **suscripciones activas** (`estado == 'activa'`). Mantenemos `fecha_fin` como `NaT` y creamos la columna booleana `sigue_activa` (`fecha_fin.isna()`).
* Normalizamos `plan` (`mensual`, `anual`), `estado` (`activa`, `cancelada`, `pausada`), `precio` (`float`) y `renovada` (`bool`).


In [4]:
MAPA_ESTADO_SUSC = {
    'activa': 'activa', 'ok': 'activa',
    'cancelada': 'cancelada', 'baja': 'cancelada',
    'pausada': 'pausada'
}
MAPA_BOOL = {
    '1': True, 'si': True, 'sí': True, 'true': True,
    '0': False, 'no': False, 'false': False
}

df_suscripciones = pd.read_csv(os.path.join(RAW_DIR, 'suscripciones.csv'), encoding='utf-8')
filas_antes = len(df_suscripciones)
df_suscripciones = df_suscripciones.drop_duplicates(subset=['suscripcion_id']).copy()

df_suscripciones['plan'] = df_suscripciones['plan'].astype(str).str.strip().str.lower().map(MAPA_PLAN)
df_suscripciones['fecha_inicio'] = parsear_fecha(df_suscripciones['fecha_inicio'])
df_suscripciones['fecha_fin'] = parsear_fecha(df_suscripciones['fecha_fin'])
df_suscripciones['precio'] = limpiar_moneda(df_suscripciones['precio'])
df_suscripciones['estado'] = df_suscripciones['estado'].astype(str).str.strip().str.lower().map(MAPA_ESTADO_SUSC)
df_suscripciones['renovada'] = df_suscripciones['renovada'].astype(str).str.strip().str.lower().map(MAPA_BOOL)
df_suscripciones['sigue_activa'] = df_suscripciones['fecha_fin'].isna()

print(f"Suscripciones: {filas_antes:,} brutas -> {len(df_suscripciones):,} limpias ({filas_antes - len(df_suscripciones):,} duplicadas eliminadas)")
print("Cruce estado vs sigue_activa (fecha_fin vacía):")
print(pd.crosstab(df_suscripciones['estado'], df_suscripciones['sigue_activa']))


Suscripciones: 51,601 brutas -> 51,243 limpias (358 duplicadas eliminadas)
Cruce estado vs sigue_activa (fecha_fin vacía):
sigue_activa  False  True 
estado                    
activa            0  17278
cancelada     30510      0
pausada        3455      0


### 4. Limpieza y Validación de `pagos.csv` (Webhook de Stripe, Nulo Real e Importes Imposibles)
* **1. Deduplicación:** Eliminamos los `1.360` pagos repetidos por `pago_id` (`171.459` $\rightarrow$ `170.099` pagos únicos).
* **2. Formato y Grafías:**
  * `importe` a `float`.
  * `tipo` normalizado a `'alta'` (`alta`, `nueva`, `inicial`) y `'renovacion'` (`renovacion`, `renovación`, `renov`, `recurrente`).
* **3. Nulo Real (`metodo`):** Los vacíos se imputan como `'desconocido'` (y las grafías a `'tarjeta'`, `'paypal'`, `'bizum'`).
* **4. Validación de Datos Imposibles / Fuera de rango (`importe_valido`):**
  * Como indica la guía: *«No los borres del fichero —son hechos reales—, pero exclúyelos del cálculo que no deben tocar. Quédate con los importes positivos y razonables (entre 0 y el precio anual, 119 €)»*. Marcamos `importe_valido = (importe > 0) & (importe <= 119.0)`.


In [5]:
MAPA_TIPO_PAGO = {
    'alta': 'alta', 'nueva': 'alta', 'inicial': 'alta',
    'renovacion': 'renovacion', 'renovación': 'renovacion', 'renov': 'renovacion', 'recurrente': 'renovacion'
}
MAPA_METODO_PAGO = {
    'tarjeta': 'tarjeta', 'card': 'tarjeta',
    'paypal': 'paypal',
    'bizum': 'bizum'
}

df_pagos = pd.read_csv(os.path.join(RAW_DIR, 'pagos.csv'), encoding='utf-8')
p_antes = len(df_pagos)
df_pagos = df_pagos.drop_duplicates(subset=['pago_id']).copy()

df_pagos['fecha'] = parsear_fecha(df_pagos['fecha'])
df_pagos['importe_raw'] = df_pagos['importe']
df_pagos['importe'] = limpiar_moneda(df_pagos['importe'])
df_pagos['tipo'] = df_pagos['tipo'].astype(str).str.strip().str.lower().map(MAPA_TIPO_PAGO)
df_pagos['metodo'] = (
    df_pagos['metodo']
    .astype(str).str.strip().str.lower()
    .map(MAPA_METODO_PAGO)
    .fillna('desconocido')
)

# Marcar importes válidos: positivos y <= 119 € (excluyendo reembolsos negativos, ceros y textos 'error'/'N/D')
df_pagos['importe_valido'] = df_pagos['importe'].notna() & (df_pagos['importe'] > 0) & (df_pagos['importe'] <= 119.0)

print(f"Pagos: {p_antes:,} brutos -> {len(df_pagos):,} únicos ({p_antes - len(df_pagos):,} duplicados de Stripe eliminados)")
print(f"Pagos con importe fuera de rango / no válido marcados (importe_valido == False): {(~df_pagos['importe_valido']).sum()}")
print(df_pagos.loc[~df_pagos['importe_valido'], ['pago_id', 'suscripcion_id', 'importe_raw', 'importe', 'tipo', 'metodo']])
df_pagos = df_pagos.drop(columns=['importe_raw'])


Pagos: 171,459 brutos -> 170,099 únicos (1,360 duplicados de Stripe eliminados)
Pagos con importe fuera de rango / no válido marcados (importe_valido == False): 17
        pago_id  suscripcion_id importe_raw  importe        tipo   metodo
1029    9163269          749173    -14,99 €   -14.99        alta  tarjeta
3659    9105688          731803      0,00 €     0.00  renovacion   paypal
26262   9128484          738622   -119,00 €  -119.00  renovacion  tarjeta
51081   9154478          746569         N/D      NaN  renovacion   paypal
51301   9100142          730097         0 €     0.00  renovacion  tarjeta
51778   9019355          705916           -      NaN        alta   paypal
66555   9079900          724200     -9,99 €    -9.99  renovacion  tarjeta
69629   9119554          736087    -14,99 €   -14.99        alta  tarjeta
74022   9062252          718923       error      NaN  renovacion   paypal
77409   9048484          714687         0 €     0.00  renovacion  tarjeta
91689   9121232       

### 5. Limpieza y Validación de `matriculas.csv` y `cursos.csv`
* En `matriculas.csv`:
  * Deduplicamos por `matricula_id` (`249.635` $\rightarrow$ `247.654` matrículas únicas).
  * Normalizamos `completado` a booleano (`True`/`False`) y `pct_completado` a `float` (0–100).
  * Validamos datos imposibles sin borrar la fila: marcamos `pct_valido` (`0 <= pct_completado <= 100`) y `minutos_validos` (`0 <= minutos_vistos <= 2000`, detectando outliers como `999999` o negativos).
* En `cursos.csv`:
  * Mapeamos las **18 grafías de `categoria`** a las **6 categorías canónicas** (`programacion`, `diseno`, `marketing`, `ofimatica`, `idiomas`, `negocios`).
  * Mapeamos `nivel` (`principiante`, `intermedio`, `avanzado`), limpiamos `duracion_horas` a `float` y marcamos `curso_valido` (`n_lecciones > 0` y `duracion_horas > 0`).


In [6]:
# 5A. matriculas.csv
df_matriculas = pd.read_csv(os.path.join(RAW_DIR, 'matriculas.csv'), encoding='utf-8')
m_antes = len(df_matriculas)
df_matriculas = df_matriculas.drop_duplicates(subset=['matricula_id']).copy()

df_matriculas['fecha_matricula'] = parsear_fecha(df_matriculas['fecha_matricula'])
df_matriculas['fecha_completado'] = parsear_fecha(df_matriculas['fecha_completado'])
df_matriculas['completado'] = df_matriculas['completado'].astype(str).str.strip().str.lower().map(MAPA_BOOL)
df_matriculas['pct_completado'] = limpiar_porcentaje_100(df_matriculas['pct_completado'])

# Validar rangos en matrículas (sin borrar las filas)
df_matriculas['pct_valido'] = df_matriculas['pct_completado'].between(0, 100)
df_matriculas['minutos_validos'] = df_matriculas['minutos_vistos'].between(0, 5000)
df_matriculas['fechas_validas'] = df_matriculas['fecha_completado'].isna() | (df_matriculas['fecha_completado'] >= df_matriculas['fecha_matricula'])

print(f"Matrículas: {m_antes:,} brutas -> {len(df_matriculas):,} únicas ({m_antes - len(df_matriculas):,} duplicadas por doble clic eliminadas)")
print("Anomalías detectadas en matrículas (marcadas, no borradas):")
print(f"  - pct_completado fuera de [0, 100]: {(~df_matriculas['pct_valido']).sum()} filas -> valores: {df_matriculas.loc[~df_matriculas['pct_valido'], 'pct_completado'].tolist()}")
print(f"  - minutos_vistos fuera de [0, 5000]: {(~df_matriculas['minutos_validos']).sum()} filas -> valores: {df_matriculas.loc[~df_matriculas['minutos_validos'], 'minutos_vistos'].tolist()}")
print(f"  - fecha_completado < fecha_matricula: {(~df_matriculas['fechas_validas']).sum()} filas")

# 5B. cursos.csv
MAPA_CATEGORIA = {
    'programacion': 'programacion', 'programación': 'programacion', 'dev': 'programacion',
    'diseno': 'diseno', 'diseño': 'diseno', 'design': 'diseno',
    'marketing': 'marketing', 'mkt': 'marketing', 'marketing digital': 'marketing',
    'ofimatica': 'ofimatica', 'ofimática': 'ofimatica', 'office': 'ofimatica',
    'idiomas': 'idiomas', 'languages': 'idiomas',
    'negocios': 'negocios', 'business': 'negocios'
}
MAPA_NIVEL = {
    'principiante': 'principiante', 'basico': 'principiante', 'básico': 'principiante',
    'intermedio': 'intermedio',
    'avanzado': 'avanzado', 'experto': 'avanzado'
}

df_cursos = pd.read_csv(os.path.join(RAW_DIR, 'cursos.csv'), encoding='utf-8')
df_cursos['titulo'] = df_cursos['titulo'].astype(str).str.strip()
df_cursos['categoria'] = df_cursos['categoria'].astype(str).str.strip().str.lower().map(MAPA_CATEGORIA)
df_cursos['nivel'] = df_cursos['nivel'].astype(str).str.strip().str.lower().map(MAPA_NIVEL)
df_cursos['duracion_horas'] = limpiar_horas(df_cursos['duracion_horas'])
df_cursos['fecha_publicacion'] = parsear_fecha(df_cursos['fecha_publicacion'])
df_cursos['curso_valido'] = (df_cursos['n_lecciones'] > 0) & (df_cursos['duracion_horas'] > 0)

print("\nCategorías limpias en cursos.csv (18 grafías -> 6 categorías):")
print(df_cursos['categoria'].value_counts())
print(f"Cursos con n_lecciones o duracion_horas <= 0 marcados: {(~df_cursos['curso_valido']).sum()} filas")
if (~df_cursos['curso_valido']).sum() > 0:
    print(df_cursos.loc[~df_cursos['curso_valido'], ['curso_id', 'titulo', 'categoria', 'n_lecciones', 'duracion_horas']])


Matrículas: 249,635 brutas -> 247,654 únicas (1,981 duplicadas por doble clic eliminadas)
Anomalías detectadas en matrículas (marcadas, no borradas):
  - pct_completado fuera de [0, 100]: 30 filas -> valores: [110, 200, 150, 110, 150, 110, 150, 110, 150, 120, 120, 150, 110, 150, 110, 200, 120, 110, 200, 120, 120, 150, 200, 120, 200, 120, 200, 120, 200, 150]
  - minutos_vistos fuera de [0, 5000]: 24 filas -> valores: [-5, 999999, -5, 999999, 999999, -5, -30, 999999, -5, -5, -30, 999999, 999999, 999999, -30, 999999, -30, 999999, 999999, -5, -30, -30, 999999, 999999]
  - fecha_completado < fecha_matricula: 6170 filas

Categorías limpias en cursos.csv (18 grafías -> 6 categorías):
categoria
programacion    100
diseno          100
marketing       100
ofimatica       100
idiomas         100
negocios        100
Name: count, dtype: int64
Cursos con n_lecciones o duracion_horas <= 0 marcados: 14 filas
     curso_id                 titulo     categoria  n_lecciones  duracion_horas
26       5027 

### 6. Exportación de las 7 Tablas Saneadas a `data/clean/`
Guardamos cada tabla en formato `.csv` y `.parquet` para alimentar las consultas SQL de DuckDB de toda la semana.


In [7]:
tablas_limpias = {
    'canales': df_canales,
    'campanas': df_campanas,
    'usuarios': df_usuarios,
    'suscripciones': df_suscripciones,
    'pagos': df_pagos,
    'matriculas': df_matriculas,
    'cursos': df_cursos
}

for nombre, df in tablas_limpias.items():
    csv_path = os.path.join(CLEAN_DIR, f"{nombre}.csv")
    pq_path = os.path.join(CLEAN_DIR, f"{nombre}.parquet")
    df.to_csv(csv_path, index=False, encoding='utf-8')
    df.to_parquet(pq_path, index=False)
    print(f"Guardado: {nombre:15s} -> {len(df):8,d} filas | { csv_path }")


Guardado: canales         ->        5 filas | data\clean\canales.csv
Guardado: campanas        ->       45 filas | data\clean\campanas.csv
Guardado: usuarios        ->   40,000 filas | data\clean\usuarios.csv


Guardado: suscripciones   ->   51,243 filas | data\clean\suscripciones.csv


Guardado: pagos           ->  170,099 filas | data\clean\pagos.csv


Guardado: matriculas      ->  247,654 filas | data\clean\matriculas.csv
Guardado: cursos          ->      600 filas | data\clean\cursos.csv


---
## La Decisión de Criterio (Día 2 Mañana): ¿Qué es el «Valor» de un Canal?

Antes de lanzar la primera consulta SQL, dejamos escrita la definición acordada con Nuria Aguilar para no convertir las métricas en una excusa:

> **Definición oficial de Valor por Canal en AulaViva:**
> *«El valor de un canal es el **LTV medio por alumno** (el importe total acumulado que deja cada alumno dado de alta en ese canal a lo largo de su vida, incluyendo primeras cuotas y renovaciones válidas), explicado por su **tasa de finalización de cursos** (si aprende) y su **tasa de renovación** (si se queda), y ponderado por su volumen mediante la **Cuota de Valor** (qué porcentaje de los ingresos totales de AulaViva aporta ese canal).*»

**Tres reglas técnicas vinculadas a esta definición:**
1. **`campana_id` vacío = alta orgánica / sin campaña:** Al cruzar altas con campañas para el CAC, se usa siempre `LEFT JOIN` desde las altas (`usuarios`) hacia los costes (`campanas`) agregados previamente por CTE.
2. **`fecha_fin` vacía = suscripción activa:** Nunca se descartan las suscripciones sin `fecha_fin`; son los alumnos que siguen activos y generan la mayor parte del LTV.
3. **Exclusión quirúrgica de datos imposibles:** En sumas de ingresos/LTV se filtran importes válidos (`importe > 0 AND importe <= 119`), y en sumas de horas de vídeo se filtran `minutos_vistos` válidos (`minutos_vistos BETWEEN 0 AND 5000`).


---
## Día 2 (Tarde): El Ranking de Superficie en SQL (La Foto de Growth)
Conectamos DuckDB a nuestras tablas limpias en `data/clean/` y construimos las consultas que mira Diego Losada cada mañana. Recuerda el método: **confirmar antes de refutar**.


In [8]:
con = duckdb.connect()
for nombre in tablas_limpias.keys():
    pq_path = os.path.join(CLEAN_DIR, f"{nombre}.parquet").replace("\\", "/")
    con.execute(f"CREATE OR REPLACE VIEW {nombre} AS SELECT * FROM '{pq_path}'")

print("Vistas SQL registradas en DuckDB:", [r[0] for r in con.execute("SHOW TABLES").fetchall()])


Vistas SQL registradas en DuckDB: ['campanas', 'canales', 'cursos', 'matriculas', 'pagos', 'suscripciones', 'usuarios']


### Consulta SQL 1: Altas por Canal y Cuota de Altas (Función de Ventana)
Contamos cuántos alumnos entraron por cada canal y qué porcentaje representan sobre los 40.000 alumnos usando `SUM(COUNT(*)) OVER ()`.


In [9]:
q_altas = """
SELECT
    canal,
    COUNT(*) AS altas,
    SUM(COUNT(*)) OVER () AS total_altas,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS cuota_altas_pct
FROM usuarios
GROUP BY canal
ORDER BY altas DESC
"""
df_sql_altas = con.execute(q_altas).df()
df_sql_altas


,canal,altas,total_altas,cuota_altas_pct
0,cupon,15200,40000.0,38.0
1,anuncios,9600,40000.0,24.0
2,seo,8000,40000.0,20.0
3,referido,4800,40000.0,12.0
4,empresas,2400,40000.0,6.0


### Consulta SQL 2: El CAC por Canal (Siempre por CTE, Nunca Cruzando `campanas` con `usuarios`)
* **La trampa mortal:** Si uniéramos `campanas` con `usuarios` directamente por `canal`, cada campaña del canal cupón (18 campañas) se multiplicaría por sus 15.200 usuarios (`18 x 15.200 = 273.600` filas), inflando el coste por 15.200 y las altas por 18.
* **La solución limpia (Patrón CTE):** Una CTE `altas` que cuenta usuarios por canal, otra CTE `costes` que suma el coste de campañas por canal, y un `LEFT JOIN` final desde `altas` (para que `seo` no desaparezca aunque no tenga campañas).


In [10]:
q_cac = """
WITH altas AS (
    SELECT
        canal,
        COUNT(*) AS n_altas
    FROM usuarios
    GROUP BY canal
),
costes AS (
    SELECT
        canal,
        COUNT(*) AS n_campanas,
        SUM(coste) AS coste_total
    FROM campanas
    GROUP BY canal
)
SELECT
    a.canal,
    a.n_altas,
    COALESCE(c.n_campanas, 0) AS n_campanas,
    ROUND(COALESCE(c.coste_total, 0), 2) AS coste_total_eur,
    ROUND(COALESCE(c.coste_total, 0) / a.n_altas, 2) AS cac_eur
FROM altas a
LEFT JOIN costes c ON a.canal = c.canal
ORDER BY cac_eur ASC
"""
df_sql_cac = con.execute(q_cac).df()
df_sql_cac


,canal,n_altas,n_campanas,coste_total_eur,cac_eur
0,seo,8000,0,0.0,0.0
1,cupon,15200,18,91200.0,6.0
2,referido,4800,7,48000.0,10.0
3,anuncios,9600,12,211200.0,22.0
4,empresas,2400,8,96000.0,40.0


### Consulta SQL 3: Las Otras Dos Caras de la Superficie — Ingreso de Primeras Cuotas y Horas de Vídeo
1. **Ingreso de primeras cuotas (`tipo = 'alta'`):** Recorremos la cadena de 3 saltos (`pagos` $\rightarrow$ `suscripciones` $\rightarrow$ `usuarios`). Aquí aparece la **primera grieta pequeña**: el cupón NO lidera el ingreso de primera cuota; lo lidera **Anuncios** (~467k € vs ~278k € del cupón), porque el cupón descuenta la primera cuota a `4,99 €`.
2. **Horas de vídeo consumidas (`SUM(minutos_vistos) / 60.0`):** El cupón vuelve a liderar (~219k horas), pero es un **engaño por puro volumen** (tiene 15.200 alumnos; por alumno o por matrícula ve mucho menos vídeo que los demás).


In [11]:
q_superficie_completa = """
WITH altas AS (
    SELECT
        canal,
        COUNT(*) AS altas,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS cuota_altas_pct
    FROM usuarios
    GROUP BY canal
),
costes AS (
    SELECT
        canal,
        SUM(coste) AS coste_total
    FROM campanas
    GROUP BY canal
),
primeras_cuotas AS (
    SELECT
        u.canal,
        COUNT(p.pago_id) AS n_pagos_alta,
        ROUND(SUM(p.importe), 2) AS ingreso_1a_cuota_eur,
        ROUND(AVG(p.importe), 2) AS ticket_medio_alta_eur
    FROM pagos p
    JOIN suscripciones s ON p.suscripcion_id = s.suscripcion_id
    JOIN usuarios u ON s.usuario_id = u.usuario_id
    WHERE p.importe_valido = TRUE
      AND p.tipo = 'alta'
    GROUP BY u.canal
),
video AS (
    SELECT
        u.canal,
        ROUND(SUM(m.minutos_vistos) / 60.0, 0) AS horas_video_total,
        ROUND((SUM(m.minutos_vistos) / 60.0) / COUNT(DISTINCT u.usuario_id), 1) AS horas_por_alumno
    FROM matriculas m
    JOIN usuarios u ON m.usuario_id = u.usuario_id
    WHERE m.minutos_validos = TRUE
    GROUP BY u.canal
)
SELECT
    a.canal,
    a.altas,
    a.cuota_altas_pct,
    ROUND(COALESCE(c.coste_total, 0) / a.altas, 2) AS cac_eur,
    pc.ingreso_1a_cuota_eur,
    pc.ticket_medio_alta_eur,
    v.horas_video_total,
    v.horas_por_alumno
FROM altas a
LEFT JOIN costes c ON a.canal = c.canal
LEFT JOIN primeras_cuotas pc ON a.canal = pc.canal
LEFT JOIN video v ON a.canal = v.canal
ORDER BY a.altas DESC
"""
df_foto_growth = con.execute(q_superficie_completa).df()
df_foto_growth


,canal,altas,cuota_altas_pct,cac_eur,ingreso_1a_cuota_eur,ticket_medio_alta_eur,horas_video_total,horas_por_alumno
0,cupon,15200,38.0,6.0,278352.24,18.31,219462.0,14.4
1,anuncios,9600,24.0,22.0,466855.05,48.63,163801.0,17.1
2,seo,8000,20.0,0.0,379915.02,47.50,148856.0,18.6
3,referido,4800,12.0,10.0,230775.27,48.08,101878.0,21.2
4,empresas,2400,6.0,40.0,114295.53,47.62,54914.0,22.9


### Consulta SQL 4: Distinguir el Contexto y el Ruido de la Historia (Categorías y Mix de Plan)
Antes de cerrar el Día 2 verificamos dos cortes adicionales:
1. **Contexto (Matrículas por categoría de curso):** El volumen de matrículas se reparte de forma equilibrada entre las 6 categorías (~16-17% cada una), sin que ninguna monopolice la plataforma.
2. **Ruido (Reparto de plan `mensual` vs `anual` por canal):** Comprobamos si el tipo de plan explica alguna diferencia entre canales. El mix sale prácticamente idéntico en los 5 canales (**~68% mensual y ~32% anual**), por lo que el plan **NO separa a los canales ni es el motor del problema**: queda descartado como ruido.


In [12]:
q_categorias = """
SELECT
    c.categoria,
    COUNT(m.matricula_id) AS n_matriculas,
    ROUND(100.0 * COUNT(m.matricula_id) / SUM(COUNT(m.matricula_id)) OVER (), 2) AS pct_matriculas
FROM matriculas m
JOIN cursos c ON m.curso_id = c.curso_id
GROUP BY c.categoria
ORDER BY n_matriculas DESC
"""
print("1) CONTEXTO — Reparto de matrículas por categoría de curso:")
display(con.execute(q_categorias).df())

q_mix_plan = """
SELECT
    canal,
    COUNT(*) AS total_usuarios,
    ROUND(100.0 * SUM(CASE WHEN plan_inicial = 'mensual' THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_mensual,
    ROUND(100.0 * SUM(CASE WHEN plan_inicial = 'anual' THEN 1 ELSE 0 END) / COUNT(*), 2) AS pct_anual
FROM usuarios
GROUP BY canal
ORDER BY total_usuarios DESC
"""
print("\n2) RUIDO DESCARTADO — Mix de plan inicial (mensual vs anual) por canal:")
display(con.execute(q_mix_plan).df())


1) CONTEXTO — Reparto de matrículas por categoría de curso:


,categoria,n_matriculas,pct_matriculas
0,negocios,41544,16.78
1,ofimatica,41431,16.73
2,marketing,41284,16.67
3,programacion,41183,16.63
4,idiomas,41127,16.61
5,diseno,41085,16.59



2) RUIDO DESCARTADO — Mix de plan inicial (mensual vs anual) por canal:


,canal,total_usuarios,pct_mensual,pct_anual
0,cupon,15200,68.07,31.93
1,anuncios,9600,67.66,32.34
2,seo,8000,68.75,31.25
3,referido,4800,68.19,31.81
4,empresas,2400,68.63,31.38


### Conclusiones al cerrar el Día 2
1. **Datos 100% saneados y auditados en `data/clean/`:**
   - 5 canales limpios, importes y fechas convertidos, `1.360` pagos duplicados de Stripe eliminados, `1.981` matrículas duplicadas y `358` suscripciones duplicadas fuera, nulos con significado preservados y anomalías imposibles marcadas para no contaminar sumas ni promedios.
2. **La foto de Growth confirma a Diego en 3 de 4 métricas superficiales (pero asoma la primera grieta):**
   - **Altas:** Gana `cupon` con **15.200 altas (38,0 % del total)**, seguido de `anuncios` (24,0 %), `seo` (20,0 %), `referido` (12,0 %) y `empresas` (6,0 %).
   - **CAC:** Gana `cupon` entre los canales de pago con **~6 € por alta** (frente a ~22 € en `anuncios`, ~40 € en `empresas` y ~10 € en recompensas de `referido`).
   - **Horas totales de vídeo:** Gana `cupon` por volumen bruto (~219k horas), aunque al dividir por alumno ya cae al último lugar.
   - **Ingreso de 1ª cuota (1ª grieta):** Gana **`anuncios`** (~467k €) y no el cupón (~278k €), porque el cupón rebaja la entrada.
3. **Plan inicial descartado como causa:** Los 5 canales tienen el mismo mix (~68% mensual / ~32% anual).
4. **Siguiente paso (Día 3):** Abrir la grieta de verdad mirando el **Acto 2 (Aprendizaje)**: ¿terminan los alumnos del cupón los cursos que empiezan, o estamos echando agua en un cubo agujereado?
